# Training dynamics

Train a small GPT, plot the loss curve, check gradient flow and
activation norms, compare learning rates.

In [ ]:
from dataclasses import replace

import mlx.core as mx

from lmxlab.data.batching import batch_iterator
from lmxlab.data.tokenizer import CharTokenizer
from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.training.callbacks import MetricsLogger
from lmxlab.training.config import TrainConfig
from lmxlab.training.trainer import Trainer

## Setup

In [ ]:
TEXT = (
    "To be or not to be that is the question "
    "whether tis nobler in the mind to suffer "
) * 50

tok = CharTokenizer(TEXT)
tokens = mx.array(tok.encode(TEXT), dtype=mx.int32)
split = int(0.9 * len(tokens))
train_tokens, val_tokens = tokens[:split], tokens[split:]

cfg = replace(gpt_tiny(), vocab_size=tok.vocab_size)
model = LanguageModel(cfg)
mx.eval(model.parameters())
print(f"{model.count_parameters():,} params, {len(train_tokens)} train tokens")

## Train

In [ ]:
trainer = Trainer(
    model,
    TrainConfig(
        learning_rate=1e-3,
        max_steps=200,
        batch_size=4,
        warmup_steps=10,
        compile_step=False,
    ),
    callbacks=[MetricsLogger(log_interval=50)],
)

history = trainer.train(
    batch_iterator(train_tokens, batch_size=4, seq_len=32, shuffle=True)
)
print(f"final loss: {history[-1]['loss']:.4f}")

## Loss curve

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_loss_curves

    fig = plot_loss_curves([h["loss"] for h in history])
    fig.show()
except ImportError:
    print("pip install matplotlib for plots")

## Gradient flow

Gradient magnitudes across layers — looking for vanishing/exploding.

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_gradient_flow

    fig = plot_gradient_flow(model)
    fig.show()
except ImportError:
    print("pip install matplotlib for plots")

## Activation norms

Should be roughly stable through layers — big jumps indicate
normalization or initialization problems.

In [ ]:
from lmxlab.analysis.activations import ActivationCapture

with ActivationCapture(model) as cap:
    model(mx.zeros((1, 16), dtype=mx.int32))

norms = cap.layer_norms()
for key, val in sorted(norms.items()):
    print(f"  {key}: {val:.4f}")

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_layer_norms

    fig = plot_layer_norms(cap.activations)
    fig.show()
except ImportError:
    print("pip install matplotlib for plots")

## Learning rate sweep

The usual story: too low is slow, too high diverges.

In [ ]:
results = {}
for lr in [1e-4, 1e-3, 1e-2]:
    mx.random.seed(42)
    m = LanguageModel(cfg)
    mx.eval(m.parameters())
    t = Trainer(
        m,
        TrainConfig(
            learning_rate=lr,
            max_steps=100,
            batch_size=4,
            warmup_steps=5,
            compile_step=False,
        ),
    )
    h = t.train(
        batch_iterator(train_tokens, batch_size=4, seq_len=32, shuffle=True)
    )
    results[f"lr={lr}"] = [s["loss"] for s in h]
    print(f"lr={lr}: final loss = {h[-1]['loss']:.4f}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    for label, losses in results.items():
        ax.plot(losses, label=label)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Learning rate comparison")
    ax.legend()
    fig.show()
except ImportError:
    print("pip install matplotlib for plots")